# Memory Duality: MC Decomposition, CCA, and Leakage Ablation

Companion notebook to `03_pca_analysis.ipynb`. Adds three pieces of direct evidence that the hybrid QRC-ESN carries two structurally distinct memory channels:

1. **Memory Capacity decomposition** — for each lag `k`, how much of `u_{t-k}` can be linearly recovered from (a) the raw sliding window `F_win`, (b) the classical reservoir state `F_esn`, (c) the quantum readout features `F_joint`. The predicted signature is a sharp cutoff at `k = window_size` for `F_win` and a smooth α-controlled decay for `F_esn`.
2. **Canonical Correlation Analysis (CCA)** — how independent the window and ESN-state subspaces are per config. Low top correlation ⇒ complementary channels, not redundant.
3. **Leakage ablation (α = 1.0)** — a pure window-only endpoint that suppresses the recurrence while keeping the quantum feature map. Shows the explicit channel's standalone ceiling per dataset.

Outputs land in `reports/figures/` with paired `_color.png` / `_bw.png` suffixes per the project convention, and in `data/` for the ablation CSV.

In [ ]:
import os, sys, json
from pathlib import Path

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

from src.data_generation import mackey_glass, generate_arma_data, generate_narma_data
from src.experiment import (
    materialize_qrc_feature_channels,
    run_qrc_experiment_with_subseeds,
)
from src.analysis import memory_capacity_spectrum, canonical_correlations
from src.visualization import get_plot_style_config, validate_plot_style

%load_ext autoreload
%autoreload 2

print('Imports OK.')

In [ ]:
PLOT_STYLE = 'color'    # flip to 'bw' for the paired grayscale pass
SEED = 2025
TRAIN_FRACTION = 0.7
FIGURES_DIR = Path('../reports/figures')
DATA_DIR = Path('../data')
RESULTS_CSV = DATA_DIR / 'results_comparative.csv'
ABLATION_CSV = DATA_DIR / 'results_alpha1_ablation.csv'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
style_cfg = get_plot_style_config(PLOT_STYLE)

DATA_PROFILES = [
    {'name': 'Mackey_Glass_(tau=17)',  'generator': mackey_glass,           'params': {'tau': 17}},
    {'name': 'Mackey_Glass_(tau=30)',  'generator': mackey_glass,           'params': {'tau': 30}},
    {'name': 'Mackey_Glass_(tau=100)', 'generator': mackey_glass,           'params': {'tau': 100}},
    {'name': 'ARMA_1_2_stochastic',    'generator': generate_arma_data,     'params': {}},
    {'name': 'NARMA10_Chaotic',        'generator': generate_narma_data,    'params': {'order': 10}},
    {'name': 'NARMA5_Chaotic',         'generator': generate_narma_data,    'params': {'order': 5}},
]

# MC decomposition probes one representative config per dataset
# (fixed window_size=10, n_layers=2) at three leakage points.
MC_WINDOW_SIZE = 10
MC_N_LAYERS = 2
MC_LEAKAGE_POINTS = [0.1, 0.5, 0.9]
MC_MAX_LAG = 2 * MC_WINDOW_SIZE

print(f'PLOT_STYLE={PLOT_STYLE}  figures_dir={FIGURES_DIR.resolve()}')

## 1. Memory Capacity decomposition

For each `(profile, leakage)` point we rebuild the three feature channels from the same trajectory the readout learned on, then probe how much of `u_{t-k}` each channel retains for `k ∈ [0, 2·window_size]`. The probe is a ridge regression with the same `lambda_reg` pattern used by the main readout.

In [ ]:
def compute_mc_for_config(profile, leakage, window_size, n_layers, seed):
    time_series = profile['generator'](**profile['params'])
    params = (leakage, 1e-8, window_size, n_layers, 0)
    channels = materialize_qrc_feature_channels(params, time_series, TRAIN_FRACTION, seed)
    ref = channels['reference_series']
    lags = np.arange(0, MC_MAX_LAG + 1)
    out = {}
    for key in ('F_win', 'F_esn', 'F_joint'):
        _, mc = memory_capacity_spectrum(channels[key], ref, lags=lags, lambda_reg=1e-8)
        out[key] = mc
    out['lags'] = lags
    out['profile'] = profile['name']
    out['leakage'] = leakage
    out['window_size'] = window_size
    return out

mc_records = []
for profile in tqdm(DATA_PROFILES, desc='MC decomposition'):
    for alpha in MC_LEAKAGE_POINTS:
        rec = compute_mc_for_config(profile, alpha, MC_WINDOW_SIZE, MC_N_LAYERS, SEED)
        mc_records.append(rec)
print(f'{len(mc_records)} MC spectra computed.')

In [ ]:
def plot_mc_panels(records, plot_style):
    style_cfg = get_plot_style_config(plot_style)
    profiles = sorted({r['profile'] for r in records}, key=lambda n: n)
    leakages = sorted({r['leakage'] for r in records})
    channel_styles = {
        'F_win':   {'label': 'Window (explicit)',   'ls': '-',  'marker': 'o'},
        'F_esn':   {'label': 'ESN state (implicit)', 'ls': '--', 'marker': 's'},
        'F_joint': {'label': 'Quantum readout',      'ls': ':',  'marker': '^'},
    }
    if plot_style == 'color':
        alpha_palette = {0.1: '#4c78a8', 0.5: '#54a24b', 0.9: '#e45756'}
    else:
        alpha_palette = {0.1: '#b0b0b0', 0.5: '#707070', 0.9: '#1a1a1a'}

    n = len(profiles)
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.8 * nrows),
                             sharey=True)
    axes = np.atleast_2d(axes)

    for idx, profile_name in enumerate(profiles):
        ax = axes[idx // ncols, idx % ncols]
        for alpha in leakages:
            rec = next((r for r in records if r['profile'] == profile_name and r['leakage'] == alpha), None)
            if rec is None:
                continue
            for channel, ch_style in channel_styles.items():
                ax.plot(rec['lags'], rec[channel],
                        linestyle=ch_style['ls'],
                        marker=ch_style['marker'], markersize=4,
                        color=alpha_palette[alpha], alpha=0.9,
                        label=f"{ch_style['label']} · α={alpha}")
        ax.axvline(MC_WINDOW_SIZE - 1, color='gray', lw=0.8, ls=':')
        ax.set_title(profile_name.replace('_', ' '), fontsize=10)
        ax.set_xlabel('lag k')
        ax.set_ylim(-0.05, 1.05)
        ax.grid(True, alpha=style_cfg['grid_alpha'])

    for j in range(n, nrows * ncols):
        axes.flatten()[j].axis('off')
    axes[0, 0].set_ylabel('MC (recoverable variance)')

    handles, labels = axes.flatten()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=8,
               bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(f'Memory capacity per channel  (window={MC_WINDOW_SIZE}, layers={MC_N_LAYERS})',
                 fontsize=12)
    fig.tight_layout(rect=[0, 0.04, 1, 0.97])
    return fig

for style in ('color', 'bw'):
    fig = plot_mc_panels(mc_records, style)
    out = FIGURES_DIR / f'memory_capacity_decomposition_{style}.png'
    fig.savefig(out, dpi=160, bbox_inches='tight')
    print(f'saved {out}')
    plt.show(); plt.close(fig)

## 2. Canonical correlations between window and ESN-state subspaces

For each QRC config in the main grid, we compute the top canonical correlation between `F_win` and `F_esn`. Low values confirm the two channels aren't redundant.

In [ ]:
qrc_rows = pd.read_csv(RESULTS_CSV)
qrc_rows = qrc_rows[qrc_rows['model_type'] == 'QRC'].copy()
# Window_size==1 collapses F_win to a single scalar channel — CCA is trivially 0
# for any non-constant reference. Keep windows >= 2.
qrc_rows = qrc_rows[qrc_rows['window_size'] >= 2]
# Sample modestly to keep the forward passes bounded. One rep per (profile, leakage, window).
sample_keys = ['data_profile', 'leakage_rate', 'window_size']
qrc_rows = qrc_rows.sort_values(['data_profile', 'leakage_rate', 'window_size', 'n_layers'])
qrc_rows = qrc_rows.drop_duplicates(subset=sample_keys, keep='last')
print(f'CCA pass over {len(qrc_rows)} unique (profile × α × window) configs.')

cca_records = []
series_cache = {p['name']: p['generator'](**p['params']) for p in DATA_PROFILES}
for _, row in tqdm(qrc_rows.iterrows(), total=len(qrc_rows), desc='CCA'):
    profile_name = row['data_profile']
    ts = series_cache.get(profile_name)
    if ts is None:
        continue
    params = (float(row['leakage_rate']), 1e-8, int(row['window_size']),
              int(row['n_layers']), int(row['lag']))
    seed = int(row['representative_seed']) if not pd.isna(row['representative_seed']) else SEED
    ch = materialize_qrc_feature_channels(params, ts, TRAIN_FRACTION, seed, include_quantum=False)
    corrs = canonical_correlations(ch['F_win'], ch['F_esn'])
    cca_records.append({
        'data_profile': profile_name,
        'leakage_rate': params[0],
        'window_size': params[2],
        'n_layers': params[3],
        'top_cca': float(corrs[0]) if corrs.size else np.nan,
        'mean_cca': float(np.mean(corrs)) if corrs.size else np.nan,
        'orthogonality_score': float(1.0 - corrs[0]**2) if corrs.size else np.nan,
    })

cca_df = pd.DataFrame(cca_records)
cca_df.head()

In [ ]:
def plot_cca_scatter(df, plot_style):
    cfg = get_plot_style_config(plot_style)
    profiles = sorted(df['data_profile'].unique())
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, prof in enumerate(profiles):
        sub = df[df['data_profile'] == prof]
        colors = [cfg['window_palette'][w] for w in sub['window_size']]
        ax.scatter(sub['leakage_rate'] + 0.01 * i, sub['top_cca'],
                   s=30 + 8 * sub['window_size'], c=colors, alpha=0.75,
                   edgecolor='black', linewidth=0.4, label=prof.replace('_', ' '))
    ax.set_xlabel('Leakage rate α')
    ax.set_ylabel('Top canonical correlation (F_win vs F_esn)')
    ax.set_ylim(0, 1.02)
    ax.axhline(1.0, color='gray', lw=0.7, ls=':')
    ax.grid(True, alpha=cfg['grid_alpha'])
    ax.legend(fontsize=8, loc='lower left', ncol=2)
    ax.set_title('Window vs ESN-state subspace alignment — lower = more orthogonal channels')
    fig.tight_layout()
    return fig

for style in ('color', 'bw'):
    fig = plot_cca_scatter(cca_df, style)
    out = FIGURES_DIR / f'window_esn_cca_{style}.png'
    fig.savefig(out, dpi=160, bbox_inches='tight')
    print(f'saved {out}')
    plt.show(); plt.close(fig)

## 3. α = 1.0 ablation (window-only endpoint)

At α = 1.0 the classical reservoir becomes `x_cl(t) = u(t)` — no recurrence. The quantum feature map still fires on the window, so this is a clean pure-explicit-memory endpoint. We compare median MSE against the best `α < 1` row per `(profile, window_size)` already present in `results_comparative.csv`.

In [ ]:
ABLATION_WINDOWS = [2, 4, 6, 8, 10]
ABLATION_N_LAYERS = 2
ABLATION_TRIALS = 11

ablation_runs = []
for profile in DATA_PROFILES:
    ts = profile['generator'](**profile['params'])
    for w in ABLATION_WINDOWS:
        params = (1.0, 1e-8, w, ABLATION_N_LAYERS, 0)
        ablation_runs.append((params, profile, ts))

def _run(item):
    params, profile, ts = item
    return run_qrc_experiment_with_subseeds(params, profile, ts, TRAIN_FRACTION, SEED,
                                            num_trials=ABLATION_TRIALS)

ablation_results = Parallel(n_jobs=-1)(
    delayed(_run)(item) for item in tqdm(ablation_runs, desc='α=1.0 ablation'))
ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(ABLATION_CSV, index=False)
print(f'saved {ABLATION_CSV}   rows={len(ablation_df)}')
ablation_df[['data_profile', 'leakage_rate', 'window_size', 'median_mse', 'std_dev_mse']].head(10)

In [ ]:
interior = qrc_rows[qrc_rows['leakage_rate'] < 1.0].copy()
best_interior = (interior
                 .sort_values(['data_profile', 'window_size', 'median_mse'])
                 .drop_duplicates(subset=['data_profile', 'window_size'], keep='first'))

comparison = ablation_df.merge(
    best_interior[['data_profile', 'window_size', 'leakage_rate', 'median_mse']]
        .rename(columns={'leakage_rate': 'best_interior_alpha',
                         'median_mse': 'best_interior_mse'}),
    on=['data_profile', 'window_size'], how='left',
)
comparison['mse_ratio'] = comparison['median_mse'] / comparison['best_interior_mse']
comparison = comparison[['data_profile', 'window_size', 'median_mse',
                         'best_interior_alpha', 'best_interior_mse', 'mse_ratio']]
comparison_path = DATA_DIR / 'ablation_alpha1_vs_interior.csv'
comparison.to_csv(comparison_path, index=False)
print(f'saved {comparison_path}')
comparison.head(20)

In [ ]:
def plot_ablation(df, plot_style):
    cfg = get_plot_style_config(plot_style)
    profiles = sorted(df['data_profile'].unique())
    fig, ax = plt.subplots(figsize=(9, 5))
    markers = ['o', 's', '^', 'D', 'v', 'P']
    for i, prof in enumerate(profiles):
        sub = df[df['data_profile'] == prof].sort_values('window_size')
        ax.plot(sub['window_size'], sub['mse_ratio'],
                marker=markers[i % len(markers)], label=prof.replace('_', ' '))
    ax.axhline(1.0, color='gray', lw=0.8, ls=':')
    ax.set_xlabel('Window size')
    ax.set_ylabel('MSE(α=1.0) / best MSE over α<1')
    ax.set_title('Window-only (α=1.0) vs. best interior leakage per (profile, window)')
    ax.grid(True, alpha=cfg['grid_alpha'])
    ax.legend(fontsize=8, loc='best', ncol=2)
    fig.tight_layout()
    return fig

for style in ('color', 'bw'):
    fig = plot_ablation(comparison, style)
    out = FIGURES_DIR / f'alpha1_ablation_{style}.png'
    fig.savefig(out, dpi=160, bbox_inches='tight')
    print(f'saved {out}')
    plt.show(); plt.close(fig)

## Expected signatures

- **MC decomposition:** `F_win` curve flat near 1 for `k < window_size` then drops to ~0; `F_esn` smooth decay whose area scales inversely with α; `F_joint` tracks the envelope of both.
- **CCA:** Top correlation well below 1 across configs (lower at low α, where ESN state differs more from the raw window).
- **α=1.0 ablation:** On chaotic Mackey-Glass (τ=30, τ=100), `mse_ratio` close to 1 — the explicit channel carries the load; on tasks where implicit dominates, the ratio grows above 1.